# Short-List Promising Models & Fine-Tune the System

## Load the Prepared Data

The preprocessor uses custom classes (`DropColumns`, `WeatherAggregator`, `ClimateFeatures`, `YearPhil`) that are defined in the Preparation notebook. `preprocessor.pkl` only stores a reference to those classes, not their code, so it can only be loaded in a notebook where they are already defined. Otherwise `pickle.load` fails with `AttributeError: module '__main__' has no attribute 'DropColumns'`.

`%run` runs the whole Preparation notebook here, which defines the classes and refits the preprocessor. It also gives this notebook:

- `X_train`, `y_train`: the preprocessed training features (57 × 160) and labels, ready for training.
- `train_set`: the raw training set, for pipelines that include `build_preprocessor()` so preprocessing is refit inside each cross-validation fold.
- `preprocessor` and `build_preprocessor()`: the fitted preprocessor and a function that builds a new unfitted one.

`X_test` and `test_set` are also loaded but should not be used until the final evaluation.

Also, `%run` needs the `nbformat` package to read notebooks. 

In [3]:
%run "Early Spring - Preparation.ipynb"

Training shape: (57, 160)
Missing values: 0
Test shape: (15, 160)
Unlabeled shape: (15, 160)
Missing values with no climate data: 0
Loaded preprocessor matches: True


In [31]:
from sklearn.metrics import f1_score, mean_squared_error, make_scorer, precision_score, recall_score
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

In [30]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "recall": make_scorer(recall_score, zero_division=0),
    "precision": make_scorer(precision_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
}


def evaluate_classifier(model):
    model.fit(X_train, y_train)
    train_predictions = model.predict(X_train)
    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
    )
    return {
        "train_f1": f1_score(y_train, train_predictions, zero_division=0),
        "cv_recall": cv_results["test_recall"].mean(),
        "cv_precision": cv_results["test_precision"].mean(),
        "cv_f1": cv_results["test_f1"].mean(),
    }

In [19]:
dt_cls = DecisionTreeClassifier(random_state=42)
dt_results = evaluate_classifier(dt_cls)
dt_results

{'train_f1': 1.0,
 'cv_recall': np.float64(0.2),
 'cv_precision': np.float64(0.16666666666666666),
 'cv_f1': np.float64(0.1688888888888889)}

In [20]:
rf_cls = RandomForestClassifier(random_state=42)
rf_results = evaluate_classifier(rf_cls)
rf_results

{'train_f1': 1.0,
 'cv_recall': np.float64(0.06666666666666667),
 'cv_precision': np.float64(0.2),
 'cv_f1': np.float64(0.1)}

In [21]:
lin_svm_cls = LinearSVC(random_state=42)
lin_svm_results = evaluate_classifier(lin_svm_cls)
lin_svm_results

{'train_f1': 1.0,
 'cv_recall': np.float64(0.4999999999999999),
 'cv_precision': np.float64(0.27666666666666667),
 'cv_f1': np.float64(0.3555555555555555)}

In [22]:
svm_rbf = SVC(kernel="rbf", random_state=42)
svm_results = evaluate_classifier(svm_rbf)
svm_results

{'train_f1': 0.5263157894736842,
 'cv_recall': np.float64(0.0),
 'cv_precision': np.float64(0.0),
 'cv_f1': np.float64(0.0)}

In [23]:
mlp_cls = MLPClassifier(hidden_layer_sizes=(50, 20), random_state=42)
mlp_results = evaluate_classifier(mlp_cls)
mlp_results

{'train_f1': 1.0,
 'cv_recall': np.float64(0.06666666666666667),
 'cv_precision': np.float64(0.06666666666666667),
 'cv_f1': np.float64(0.06666666666666667)}

In [28]:
logred_cls = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
logred_results = evaluate_classifier(logred_cls)
logred_results

{'train_f1': 1.0,
 'cv_recall': np.float64(0.29999999999999993),
 'cv_precision': np.float64(0.36666666666666664),
 'cv_f1': np.float64(0.32)}

In [32]:
# Gradient Boosting Classifier
gb_cls = GradientBoostingClassifier(random_state=42)
gb_results = evaluate_classifier(gb_cls)
gb_results

{'train_f1': 1.0,
 'cv_recall': np.float64(0.4),
 'cv_precision': np.float64(0.3466666666666667),
 'cv_f1': np.float64(0.36095238095238097)}

## Shortlist Findings

The base models were compared using five-fold stratified cross-validation. The reported `cv_recall`, `cv_precision`, and `cv_f1` values are averages across the validation folds for the positive class, `True` (early Spring). The `train_f1` value is only included to check for overfitting and is not used to select the models.

| Model | CV recall | CV precision | CV F1 | Training F1 |
|---|---:|---:|---:|---:|
| Decision tree | 0.20 | 0.167 | 0.169 | 1.000 |
| Random forest | 0.067 | 0.200 | 0.100 | 1.000 |
| Linear SVM | 0.500 | 0.277 | 0.356 | 1.000 |
| RBF SVM | 0.000 | 0.000 | 0.000 | 0.526 |
| MLP neural network | 0.067 | 0.067 | 0.067 | 1.000 |
| Logistic Regression | 0.300 | 0.370 | 0.320 | 1.000 |
| Gradient Boosting | 0.400 | 0.347 | 0.361 | 1.000 |

Because missing an early Spring is the most costly error, recall is the primary comparison metric. The Linear SVM has the strongest recall and F1, and Logistic Regression has the strongest precision and the second highest F1. The large gap between training F1 and cross-validation F1 for most models indicates overfitting, so the cross-validation results are more informative than training performance. The estimates are uncertain because the training set contains only 14 early-Spring examples. Each validation fold contains only about two or three positive examples, so the best model may change during fine-tuning.

## Models for Fine-Tuning

The three models selected to continue with are:

1. **LinearSVC:** Best early-Spring recall and best cross-validation F1.
2. **LogisticRegression:** Best precision and a competitive F1 score. Its `class_weight="balanced"` setting is used to addresses the class imbalance.
3. **RandomForestClassifier:** Nonlinear ensemble comparison, even though its recall is currently weak.

During fine-tuning, these models should be evaluated using recall, precision, and F1 for the early-Spring class. Gradient Boosting will also be tuned as the additional ensemble model.

For fine-tuning, preprocessing should be included inside each model pipeline and refit within every cross-validation fold. The current shortlist evaluates the already-preprocessed `X_train`; the next step should instead use `train_set`, `build_preprocessor()`, and a pipeline so imputation and scaling do not use information from the validation fold.

In [ ]:
# Feature Importance for Random Forest
for feature, importance in sorted(zip(rf_cls.feature_names_in_, rf_cls.feature_importances_), key=lambda x: x[1], reverse=True):
    print(f"{feature}: {importance}")

min_soil_temp_mar1_minus05: 0.02564635132765923
precipitation_mar1_minus02: 0.025445107882386967
max_soil_temp_jan_mean: 0.02389995846804874
min_temp_feb_mean: 0.02157196186098368
mean_soil_temp_mar1_minus06: 0.02079978903240463
mean_soil_temp_mar1_minus10: 0.01858375390241067
mean_soil_moisture_mar1_minus10: 0.017605468750651703
snowfall_feb_total: 0.016184123232103654
avg_temp_mar1_minus05: 0.014859969985787534
nao_dec: 0.014660706902027419
min_temp_mar1_minus05: 0.014606402074852576
max_soil_moisture_feb_mean: 0.013908535146452135
max_soil_temp_mar1_minus03: 0.01388348061383105
max_temp_mar1_minus01: 0.013872959924516117
mean_soil_moisture_feb_mean: 0.013776033413937825
max_soil_temp_mar1_minus06: 0.01230562664182262
avg_temp_mar1_minus06: 0.012041465875001134
mean_soil_temp_mar1_minus05: 0.011931082711273279
mean_soil_temp_mar1_minus02: 0.011931019579117297
min_soil_temp_mar1_minus06: 0.011198143960510063
above_freezing_feb_pct: 0.011187998330708175
min_soil_moisture_mar1_minus07: 

## Fine-Tuning The System